# Session 12: Cleaning Messy Data

Data Science with Python, Level 1, Stage 2: Working with Data

## Learning Objectives

By the end of this session you will be able to:

1. Spot problems with `info`, `isna` and `value_counts`.
2. Remove duplicate rows.
3. Standardise messy text columns.
4. Convert columns to the right types.
5. Handle missing values by dropping or filling.
6. Find and fix impossible values.

## Where We Are

The cafe data of Sessions 11 and 11 was clean: no gaps, no typos, every number stored as a number. Real data almost never looks like that. People skip questions, type cities in different ways, write `twenty` instead of `20`, and systems export the same row twice.

Data scientists often say that **80 percent of the work is cleaning**. A model or a chart built on dirty data gives wrong answers with full confidence. Today we clean a practice survey of 126 responses that contains all the classic problems.

In [ ]:
import pandas as pd
city = pd.Series(["Damascus", "damascus ", "DAMASCUS", "Aleppo"])
print(city.value_counts())

## Setup

Run this cell first. It loads the libraries and sets the address of the course datasets.

In [ ]:
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
survey = pd.read_csv(DATA_URL + "messy_survey.csv")

## 1. Spotting the Problems

In [ ]:
survey[["respondent", "age", "city", "monthly_income", "satisfaction"]].head(6)

In [ ]:
survey.dtypes

In [ ]:
survey.isna().sum()

In [ ]:
print(survey["city"].value_counts(dropna=False))
print("Duplicate rows:", survey.duplicated().sum())

## 2. Removing Duplicates

In [ ]:
print(len(survey))
survey = survey.drop_duplicates()
print(len(survey))
print(survey["respondent"].duplicated().sum(), "repeated IDs left")

## 3. Cleaning Text

In [ ]:
survey["city"] = survey["city"].str.strip().str.title()
print(survey["city"].value_counts(dropna=False))

In [ ]:
fixes = {"Halab": "Aleppo", "Sham": "Damascus"}
cleaned = pd.Series(["Halab", "Homs", "Sham"]).replace(fixes)
print(cleaned.tolist())

## 4. Fixing Types

`pd.to_numeric` converts a column to numbers. With `errors="coerce"`, any value that cannot be converted becomes `NaN`, the missing value marker, instead of stopping with an error. Always look at **which** values failed before you decide what to do with them.

In [ ]:
age_num = pd.to_numeric(survey["age"], errors="coerce")
failed = survey.loc[age_num.isna(), "age"]
print(failed.tolist())

In [ ]:
age_text = survey["age"].astype(str).str.strip()
age_text = age_text.str.replace("years", "").str.replace("yrs", "")
age_text = age_text.replace({"thirty": "30"})
survey["age"] = pd.to_numeric(age_text, errors="coerce")
print(survey["age"].dtype, survey["age"].isna().sum(), "missing")

In [ ]:
survey["satisfaction"] = pd.to_numeric(survey["satisfaction"].replace({"five": "5"}), errors="coerce")
survey["signup_date"] = pd.to_datetime(survey["signup_date"])
print(survey[["age", "satisfaction", "signup_date"]].dtypes)

## 5. Handling Missing Values

There is no single right answer for missing values. **Dropping** rows is simple but throws information away. **Filling** keeps the rows, but the filled value is a guess. Common choices: fill numbers with the **median**, which is not pulled by extreme values, and fill categories with the most common value or a label such as `Unknown`. Write down every choice you make.

In [ ]:
print(survey.isna().sum()[lambda s: s > 0])

survey["city"] = survey["city"].fillna("Unknown")
median_income = survey["monthly_income"].median()
survey["monthly_income"] = survey["monthly_income"].fillna(median_income)
survey = survey.dropna(subset=["satisfaction"])
print(survey.isna().sum().sum(), "missing values left besides age and hours:",
      survey[["age", "hours_online"]].isna().sum().tolist())

| Situation | Usual choice | Code |
| --- | --- | --- |
| Very few rows missing a key value | Drop those rows | `df.dropna(subset=["col"])` |
| A number missing in many rows | Fill with the median | `df["col"].fillna(df["col"].median())` |
| A category missing | Fill with a label | `df["col"].fillna("Unknown")` |
| The target you want to predict | Drop those rows | `df.dropna(subset=["target"])` |
| A column mostly empty | Drop the column | `df.drop(columns=["col"])` |

## 6. Impossible Values

In [ ]:
bad_age = ~survey["age"].between(16, 100)
print(survey.loc[bad_age, ["respondent", "age"]])

survey.loc[bad_age, "age"] = np.nan
survey["age"] = survey["age"].fillna(survey["age"].median())

survey.loc[survey["hours_online"] > 24, "hours_online"] = np.nan
survey["hours_online"] = survey["hours_online"].fillna(survey["hours_online"].median())
print(survey[["age", "hours_online"]].describe().loc[["min", "max"]])

In [ ]:
assert survey.duplicated().sum() == 0
assert survey.isna().sum().sum() == 0
assert survey["age"].between(16, 100).all()
print(survey.shape)
print(survey["city"].value_counts())

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Change lost | Calling `df.drop_duplicates()` without storing it | Write `df = df.drop_duplicates()`. |
| Data hidden | Using `errors="coerce"` without looking at what failed | List the failed values first, then decide. |
| Skewed fill | Filling numbers with the mean when there are outliers | Use the median, which outliers do not pull. |
| Too much dropped | Calling `dropna()` on the whole table | Drop by column with `subset=[...]`, or fill. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Count the Problems

Load the raw survey again into a new variable. Print the number of rows, duplicates, missing values per column, and the number of different spellings of the city column, including missing values.

In [ ]:
# Write your code below

### In-Class Exercise 2: Clean Names

Clean this Series so every name has no extra spaces and capital first letters: `pd.Series(["  lina haddad", "OMAR SALEH ", "sara  nassar"])`. Hint: `str.split().str.join(" ")` also removes double spaces in the middle.

In [ ]:
# Write your code below

### In-Class Exercise 3: Prices as Text

Convert `pd.Series(["12.5", "8", "$15", "n/a", "7,5"])` to numbers. Repair the dollar sign and the comma first, and count how many values remain missing.

In [ ]:
# Write your code below

### In-Class Exercise 4: Fill Smartly

For `pd.DataFrame({"group": ["A", "A", "B", "B", "B"], "score": [10, None, 30, 34, None]})`, fill each missing score with the median score of its own group. Hint: `df.groupby("group")["score"].transform("median")` gives each row its group's median.

In [ ]:
# Write your code below

## Summary

- Look first: `dtypes`, `isna().sum()`, `value_counts`, `duplicated`.
- `drop_duplicates` keeps one copy of each record.
- `.str.strip().str.title()` standardises text.
- `pd.to_numeric(errors="coerce")` reveals values that are not numbers.
- Drop or fill missing values, column by column.
- Turn impossible values into missing ones, then test with `assert`.

## What's Next

With clean data we can finally compare groups: revenue per branch, salary per department, satisfaction per city. Next session `groupby` answers these questions in one line each.